In [37]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "sanchez2018chimpanzees")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "sanchez_2018_Data_all models_updated_final.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [38]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)


df['study_id']="sanchez2018chimpanzees"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


In [39]:
df = df.rename(columns={"sex-dyad": "dyad_sex",
    "dyad-sex":"dyad_sex",
    "ind_l": "individual_left",
    "ind_r": "individual_right",
    "subj_lat_opendoor":"subject_lat_opendoor",
    "part_lat_opendoor":"partner_lat_opendoor",
    "subj_lat_doortopull":"subject_lat_doortopull",
    "part_lat_doortopull":"partner_lat_doortopull",
    "subj_lat_doorandpull":"subject_lat_doorandpull",
    "part_lat_doorandpull":"parter_lat_doorandpull"})
df['subject'] = df['subject'].str.rstrip()
df['partner'] = df['partner'].str.rstrip()

In [40]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
for x,y in zip(df_name['wrong'],df_name['right']):
    df['subject'].replace(x, y, inplace=True)
    df['partner'].replace(x, y, inplace=True)

In [41]:
df["subject"]=df["subject"].replace("na", np.nan, regex=True)
df["partner"]=df["partner"].replace("na", np.nan, regex=True)

# df['dyad']=df.subject.str.cat(df.partner, sep='_')


In [42]:
df=df.rename(columns={"subject":"ape",
    "partner":"ape_2"})


In [43]:
role=[]
role_2=[]
for index, row in df.iterrows():
    if not pd.isna(row['ape']):
        role.append("subject")
    else:
        role.append("")
df = df.assign(role=role)
for index, row in df.iterrows(): 
    if not pd.isna(row['ape_2']):
        role_2.append("partner")
    else:
        role_2.append("")
df = df.assign(role_2=role_2)


comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

df=df.rename(columns={"species":"species_subject"})
df['species']='chimpanzee'

In [44]:
df.replace('na', np.nan, inplace=True)
# df.columns
df.rename(columns={"ape": "participant", "ape_2":"participant_2",
                   'side_subject':'side_focal_participant',
                   'subject_decision':'focal_participant_decision',
                   'subject_lat_opendoor':'focal_participant_lat_opendoor', 
                   'subject_lat_doortopull':'focal_participant_lat_doortopull',
                   'subject_success':'focal_participant_success',
                   'subject_lat_doorandpull':'focal_participant_lat_doorandpull',
                   'subject_latency_total':'focal_participant_latency_total',
                   'soc_choice(0 is yes)':'soc_choice',
                   'cox1_freechoicesub':'cox1_freechoice_focal'}, inplace=True)

df['role'].replace('subject', 'focal_participant', inplace=True, regex=True)

In [45]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
df= df.merge(ape_dob,left_on='participant', right_on='name', how='left')

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
df= df.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')
two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    df[x] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
    df[x] = pd.to_datetime(df[x])
    df[y] = pd.to_datetime(df[y])
    df[k] = (df[x] - df[y]).dt.days//365

In [46]:
sanchez2018chimpanzees_standardized=df[['study_id', 'year','month',  'day', 
        'participant','age_in_years','sex','role', 'participant_2','age_in_years_2',  'sex_2', 'role_2',  'species',
        'dyad','dyad_sex', 
       'phase','session','trial',  'condition', 'leverage', 
       'side_focal_participant', 'side_partner', 'individual_left', 'individual_right',
       '1st_access', 'focal_participant_decision', 'partner_decision',
       'focal_participant_lat_opendoor', 'partner_lat_opendoor',
       'focal_participant_lat_doortopull', 'partner_lat_doortopull', 'focal_participant_success',
       'partner_success', 'lev_underst_full', 'lev_underst_gen',
       'focal_participant_lat_doorandpull', 'parter_lat_doorandpull',
       'focal_participant_latency_total', 'partner_latency_total',
       'soc_choice', 'cox1_freechoice_focal',
       'cox2_freechoicepull', 'cox3_freechoice' ]]


In [47]:
comp_out_path_stand = os.path.join(out_pathway, 'sanchez2018chimpanzees_standardized.csv')
sanchez2018chimpanzees_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names =sanchez2018chimpanzees_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
sanchez2018chimpanzees_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'sanchez2018chimpanzees_glossary.csv')
sanchez2018chimpanzees_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)